In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 1 · [BOOTSTRAP]  Auto-setup: Local Windows + Google Cloud (GCE)     ║
# ║  Run this cell FIRST every session — clones/pulls repo & installs deps.   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import os, sys, subprocess, platform
from pathlib import Path

GITHUB_REPO   = 'https://github.com/DhruvalPtl/alpha-synuclein-agent.git'
CLOUD_INSTALL = Path.home() / 'agent_workspace'
LOCAL_WIN     = Path(r'd:\3rd sem M.tech\agent_workspace')

def _on_cloud():
    try:
        import urllib.request
        req = urllib.request.Request(
            'http://metadata.google.internal/',
            headers={'Metadata-Flavor': 'Google'}
        )
        urllib.request.urlopen(req, timeout=0.8)
        return True
    except Exception:
        pass
    return not LOCAL_WIN.exists()

IS_CLOUD = _on_cloud()
print(f'Environment : {"GOOGLE CLOUD (GCE)" if IS_CLOUD else "LOCAL"}')
print(f'Platform    : {platform.system()} {platform.machine()}')
print(f'Python      : {sys.version.split()[0]}  ({sys.executable})')

if IS_CLOUD:
    if not CLOUD_INSTALL.exists():
        print(f'\nCloning {GITHUB_REPO} -> {CLOUD_INSTALL} ...')
        r = subprocess.run(['git', 'clone', GITHUB_REPO, str(CLOUD_INSTALL)],
                           text=True, capture_output=True)
        if r.returncode != 0:
            raise RuntimeError(f'git clone failed:\n{r.stderr}')
        print('Clone complete.')
    else:
        print(f'\nRepo found at {CLOUD_INSTALL} — pulling latest ...')
        r = subprocess.run(['git', '-C', str(CLOUD_INSTALL), 'pull'],
                           text=True, capture_output=True)
        print(r.stdout.strip() or r.stderr.strip())
        # Rebuild leaderboard after pull (merges experiments from both machines)
        print('Rebuilding leaderboard from disk after pull ...')
        rb_r = subprocess.run(
            [sys.executable, '-m', 'agent.tools.rebuild_leaderboard', '--quiet'],
            capture_output=True, text=True, cwd=str(CLOUD_INSTALL)
        )
        if rb_r.returncode == 0:
            print('Leaderboard rebuilt.')
        else:
            print(f'Leaderboard rebuild skipped: {rb_r.stderr[:100]}')
    PROJECT_ROOT = CLOUD_INSTALL
else:
    PROJECT_ROOT = LOCAL_WIN
    print(f'\nLocal workspace: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Working dir : {os.getcwd()}')

def _has_nvidia():
    try:
        return subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

if IS_CLOUD and _has_nvidia():
    print('\nGPU detected — installing PyTorch CUDA 12.1 ...')
    r = subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'torchvision', 'torchaudio',
        '--index-url', 'https://download.pytorch.org/whl/cu121'
    ], capture_output=True, text=True)
    print('PyTorch CUDA OK.' if r.returncode == 0
          else f'CUDA wheel failed, CPU fallback: {r.stderr[:200]}')
elif IS_CLOUD:
    print('\nNo GPU — CPU PyTorch will install via requirements.')

print('\nInstalling requirements.txt ...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     str(PROJECT_ROOT / 'requirements.txt')],
    text=True, capture_output=True
)
if r.returncode != 0:
    print('[STDERR]', r.stderr[-2000:])
    raise RuntimeError('pip install failed — see above')
print('Requirements OK.')

for d in ['master_log', 'data/raw', 'data/splits', 'data/processed', 'experiments']:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

env_path = PROJECT_ROOT / '.env'
if not env_path.exists():
    env_path.write_text(
        '# Fill in the API keys you want to use\n'
        'GEMINI_API_KEY=\nGROQ_API_KEY=\nMISTRAL_API_KEY=\n'
        'OPENROUTER_API_KEY=\nCEREBRAS_API_KEY=\n'
        '# Set your machine tag (avoids experiment ID collisions across machines)\n'
        'MACHINE_ID=gcloud\n'
    )
    print(f'\nCreated .env at {env_path}  <-- edit MACHINE_ID and API keys!')
else:
    print(f'\n.env found at {env_path}')

if IS_CLOUD and _has_nvidia():
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'\nGPU : {r.stdout.strip()}')

# Show current MACHINE_ID so user knows what tag their experiments will get
import os as _os
machine_id = _os.environ.get('MACHINE_ID', '') or platform.node()
print(f'\nMACHINE_ID  : {machine_id}  (experiments tagged with this)')

print('\n' + '='*60)
print('  BOOTSTRAP COMPLETE')
print(f'  Root    : {PROJECT_ROOT}')
print(f'  Cloud   : {IS_CLOUD}')
print('  --> Run Cell 2 to build / verify the data pipeline.')
print('='*60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 2 · [VERIFY DATA]  Load CSV -> features -> reduce -> splits -> save  ║
# ║  Safe to re-run: skips expensive feature engineering if splits exist.      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import numpy as np
from pathlib import Path
from agent.data.pipeline import DataPipeline
from agent.core.tee_logger import TeeLogger

logger = TeeLogger(master_log_dir='master_log')
logger.info('=== Phase 1: Data Verification Start ===')

pipe = DataPipeline(splits_dir='data/splits', random_state=42)

_train_pkl = Path('data/splits/train.pkl')
_selector  = Path('data/processed/selector.pkl')
_weights   = Path('data/processed/class_weights.pkl')

if _train_pkl.exists() and _selector.exists() and _weights.exists():
    logger.info('Existing pipeline artifacts found — loading.')
    pipe.load_splits()
else:
    csv_path = Path('data/raw/alpha_synuclein.csv')
    if not csv_path.exists():
        raise FileNotFoundError(
            f'CSV not found: {csv_path}\n'
            'It is committed to GitHub — check that the clone succeeded.'
        )
    logger.info('Loading and expanding CSV ...')
    df = pipe.load_and_expand(str(csv_path))
    logger.info(f'Expanded DataFrame: {df.shape}')

    logger.info('Engineering features (2-mer + 3-mer) ...')
    X, y = pipe.build_features(df)
    logger.info(f'Raw feature matrix: {X.shape}')

    logger.info('Stratified split 70/15/15 ...')
    pipe.stratified_split(X, y, train=0.70, val=0.15, test=0.15)

    logger.info('Reducing features (VarianceThreshold + SelectKBest k=500) ...')
    pipe.reduce_features(
        pipe.X_train, pipe.X_val, pipe.X_test, pipe.y_train, k_best=500
    )

    logger.info('Computing class weights ...')
    pipe.get_class_weights(pipe.y_train)

    logger.info('Saving splits ...')
    pipe.save_splits()

print()
label_names = {0: 'No', 1: 'Low', 2: 'Medium', 3: 'High'}
unique, counts = np.unique(pipe.y_train, return_counts=True)
print('='*52)
print(f'Train : {pipe.X_train.shape}  Val: {pipe.X_val.shape}  Test: {pipe.X_test.shape}')
for u, c in zip(unique.tolist(), counts.tolist()):
    print(f'  Class {u} ({label_names[u]:<6}): {c} train samples')
print('data/processed/ artifacts:')
for f in sorted(Path('data/processed').glob('*.pkl')):
    print(f'  {f.name}  ({f.stat().st_size:,} B)')
print('='*52)
logger.info('=== Phase 1: Data Verification Complete ===')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 3 · [VERIFY WALL]  Seal and verify the mathematical wall             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
from agent.core.tee_logger import TeeLogger

logger = TeeLogger(master_log_dir='master_log')

logger.info('Sealing test set ...')
digest = pipe.seal_test_set()

logger.info('Verifying mathematical wall integrity ...')
wall_intact = pipe.verify_wall()

print()
print('='*60)
print(f'  SHA-256 : {digest}')
print(f'  Wall    : {"INTACT" if wall_intact else "*** BREACH DETECTED ***"}')
print('='*60)
print()
print('CROSS-MACHINE NOTE:')
print('  Copy the SHA-256 above and compare it to the hash')
print('  produced by running this cell on the cloud instance.')
print('  Matching = reproducible test sets. Different = invalid comparisons.')

if wall_intact:
    logger.info('Mathematical wall INTACT. Safe to proceed.')
else:
    logger.error('WALL BREACH — test set has been tampered with!')
    raise RuntimeError('Wall breach detected. Do not run experiments.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 4 · [LAUNCH]  Start the autonomous ML research agent                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
from agent.core.orchestrator import AgentOrchestrator

# ── Choose your LLM — change this ONE line ────────────────────────────────────
# Local (Ollama must be running on the instance):
# agent = AgentOrchestrator(model_name='local-qwen')

# Cloud API (fill .env with key first):
agent = AgentOrchestrator(model_name='groq-llama')     # GROQ_API_KEY
# agent = AgentOrchestrator(model_name='gemini-flash') # GEMINI_API_KEY
# agent = AgentOrchestrator(model_name='mistral-small')# MISTRAL_API_KEY

print('Agent status:', agent.status())
print()
print('Starting autonomous research loop (max 200 experiments) ...')
print('Interrupt kernel at any time to stop gracefully.')
print()

agent.run(max_experiments=200)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  Cell 5 · [DASHBOARD]  Live combined leaderboard — both machines          ║
# ║  Calls rebuild_leaderboard() before each render so it reflects all        ║
# ║  experiments from BOTH laptop and cloud (via synced results.json files).  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import json, time, subprocess, threading
from pathlib import Path
from IPython.display import clear_output, display, HTML
import ipywidgets as widgets

from agent.tools.rebuild_leaderboard import rebuild_leaderboard

_STATE_PATH = Path('master_log/orchestrator_state.json')
_REFRESH_S  = 30

def _gpu_info():
    try:
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=3
        )
        if r.returncode == 0:
            used, total, util = r.stdout.strip().split(',')
            return f'GPU {used.strip()}/{total.strip()} MB | {util.strip()}% util'
    except Exception:
        pass
    return 'GPU: n/a'

def _read_state():
    try:
        return json.loads(_STATE_PATH.read_text()) if _STATE_PATH.exists() else {}
    except Exception:
        return {}

def _bar(done, total=200, w=40):
    f = int(w * done / max(total, 1))
    return f'[{"#"*f + "-"*(w-f)}] {done}/{total} ({100*done/max(total,1):.1f}%)'

def _render(out):
    # Always rebuild from disk first — merges both machines' results
    try:
        lb = rebuild_leaderboard(verbose=False)
    except Exception as exc:
        lb = {}
        print(f'rebuild_leaderboard error: {exc}')

    state = _read_state()
    exps  = lb.get('experiments', [])
    n     = lb.get('total_runs', 0)
    f1    = lb.get('best_val_f1_macro', 0.0)
    best  = lb.get('best_experiment', 'none')
    fams  = lb.get('families_completed', [])
    upd   = lb.get('last_updated', 'never')
    mach  = lb.get('runs_per_machine', {})
    st    = state.get('status', 'unknown')
    model = state.get('model', '?')
    sc    = {'running':'#66bb6a','idle':'#bdbdbd','stopping':'#ffa726'}.get(st,'#ef5350')

    # Machine breakdown badge
    mach_str = '  '.join(f'{m}:{c}' for m, c in sorted(mach.items()))

    rows = exps[:10]  # already sorted by f1 desc
    tbl  = '<table style="border-collapse:collapse;width:100%;font-size:.84em">'
    tbl += '<tr style="background:#1e2a32">'
    for h in ['#','ID','Architecture','Family','Machine','F1-macro','Acc','Time','Status']:
        tbl += f'<th style="padding:4px 8px;color:#90caf9;text-align:left">{h}</th>'
    tbl += '</tr>'
    for i, e in enumerate(rows, 1):
        v  = e.get('val_f1_macro',0)
        vc = '#66bb6a' if v>=.6 else ('#ffa726' if v>=.3 else '#ef5350')
        sc2= '#66bb6a' if e.get('status')=='success' else '#ef5350'
        bg = '#131c22' if i%2 else '#0d1117'
        mid = e.get('machine_id','?')
        mc  = '#80cbc4' if mid == 'gcloud' else '#ffcc80'
        tbl += (
            f'<tr style="background:{bg}">'
            f'<td style="padding:3px 8px;color:#78909c">{i}</td>'
            f'<td style="padding:3px 8px;color:#b0bec5;font-size:.78em">{e.get("exp_id","?")[:35]}</td>'
            f'<td style="padding:3px 8px;color:#e0e0e0">{e.get("architecture","?")[:20]}</td>'
            f'<td style="padding:3px 8px;color:#80cbc4">{e.get("architecture_family","?")[:13]}</td>'
            f'<td style="padding:3px 8px;color:{mc};font-size:.78em">{mid}</td>'
            f'<td style="padding:3px 8px;font-weight:bold;color:{vc}">{v:.4f}</td>'
            f'<td style="padding:3px 8px;color:#78909c">{e.get("val_accuracy",0):.4f}</td>'
            f'<td style="padding:3px 8px;color:#78909c">{e.get("train_time_seconds",0):.1f}s</td>'
            f'<td style="padding:3px 8px;color:{sc2}">{e.get("status","?")}</td>'
            f'</tr>'
        )
    tbl += '</table>'

    ALL = ['classical_ml','linear','neural_network','deep_residual',
           'ensemble_stack','attention_based','graph_neural','automl']
    pills = ''.join(
        f'<span style="background:{"#2e7d32" if fa in fams else "#37474f"};'
        f'color:#e0e0e0;border-radius:4px;padding:2px 8px;margin:2px;'
        f'font-size:.8em;display:inline-block">{"v" if fa in fams else "."} {fa}</span>'
        for fa in ALL
    )

    html = (
        f'<div style="background:#0d1117;color:#e0e0e0;padding:16px;border-radius:8px;font-family:monospace">'
        f'<h2 style="margin:0 0 4px;color:#4fc3f7">Alpha-Synuclein Agent — Combined Leaderboard</h2>'
        f'<small>Rebuilt from disk each refresh | Updated: {upd}</small><br><br>'
        f'<b>Status:</b> <span style="color:{sc}">{st.upper()}</span>&nbsp;&nbsp;'
        f'<b>Model:</b> {model}&nbsp;&nbsp;<b>{_gpu_info()}</b><br>'
        f'<b>Progress:</b> {_bar(n)}&nbsp;&nbsp;'
        f'<b>Machines:</b> <span style="color:#80cbc4">{mach_str or "none yet"}</span><br>'
        f'<b>Best F1-macro:</b> <span style="font-size:1.3em;color:#ffb74d">{f1:.4f}</span>'
        f' ({best})<br><br>'
        f'{tbl}<br>'
        f'<b>Families:</b><br>{pills}'
        f'</div>'
    )
    with out:
        clear_output(wait=True)
        display(HTML(html))

out      = widgets.Output()
_stop    = [False]
btn_stop = widgets.Button(description='Stop', button_style='danger', icon='stop')
btn_now  = widgets.Button(description='Refresh', button_style='info', icon='refresh')

btn_stop.on_click(lambda _: [_stop.__setitem__(0, True),
                              btn_stop.__setattr__('disabled', True)])
btn_now.on_click(lambda _: _render(out))

display(widgets.HBox([btn_now, btn_stop]))
display(out)
_render(out)

def _loop():
    while not _stop[0]:
        for _ in range(_REFRESH_S * 2):
            if _stop[0]: return
            time.sleep(0.5)
        _render(out)

threading.Thread(target=_loop, daemon=True).start()
print(f'Dashboard running — rebuilt from disk every {_REFRESH_S}s.')